In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler

# Diabetes prediction - Preprocessing

## Load data

In [2]:
diabetes_data = pd.read_csv("../data/processed/diabetes_data_cleaned.csv")

In [3]:
diabetes_data

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0
...,...,...,...,...,...,...,...,...,...
96123,Female,36.0,0,0,No Info,24.60,4.8,145,0
96124,Female,2.0,0,0,No Info,17.37,6.5,100,0
96125,Male,66.0,0,0,past_smoker,27.83,5.7,155,0
96126,Female,24.0,0,0,never,35.42,4.0,100,0


## Train/Test Split

We start with the splitting of the data into training and testing split. This is done before the encoding and the scaling in order to avoid data leakage.

In [4]:
X = diabetes_data.drop(columns=["diabetes"])
y = diabetes_data["diabetes"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
print(f"Train shape: {X_train.shape}, positive rate: {y_train.mean():.4f}")
print(f"Test shape: {X_test.shape}, positive rate: {y_test.mean():.4f}")

Train shape: (76902, 8), positive rate: 0.0882
Test shape: (19226, 8), positive rate: 0.0882


## Encoding

This dataset does not need much encoding. Only the `gender` and `smoking_history` will need to be encoded using one-hot-encoding.

`gender` and `smoking_history` are both nominal categorical variables - there is no inherent order between their categories, so one-hot encoding is used instead of ordinal/label encoding, which would otherwise imply a false ranking.

`smoking_history` in particular cannot be meaningfully treated as ordinal: besides the ambiguity between `never`, `current`, and `past_smoker` (a long-quit light smoker vs a current heavy smoker do not have an obvious risk ordering), the category also includes `No Info`, which represents missing data rather than a smoking status and therefore has no valid position on any risk scale. This was already noted in the EDA notebook.

## Scaling

The EDA notebook showed that the four numeric features do not all share the same distribution shape. `bmi` is heavily concentrated around a single value with a long tail of extreme outliers, and `blood_glucose_level` shows a similar, though milder, tail of higher values beyond its main discretized range. `age` and `HbA1c_level` show no comparable outlier-driven tail.

Because of this, two different scalers are used: `RobustScaler` for `bmi` and `blood_glucose_level`, since it scales using the median and interquartile range rather than the mean and standard deviation, making it less sensitive to the outliers identified in these two features. `StandardScaler` is used for `age` and `HbA1c_level`, where no such outlier concern 
applies.

Scaling matters mainly for Logistic Regression, since its regularization term penalizes coefficients on a comparable basis regardless of a feature's natural numeric range. Random Forest is scale-invariant, so scaling does not affect them, but the same scaled features are reused for all models for simplicity. `hypertension` and `heart_disease` are left unscaled, since their 0/1 values already represent category membership rather than a quantity. The same applies to `gender` and `smoking_history` one-hot encoded below.

## Applying Encoding and Scaling Together

Encoding and scaling are combined into a single `make_column_transformer`, so both are fit once on the training data and applied consistently to the test data in one step. The transformer is fit only on `X_train` (`fit_transform`) and only applied - not re-fit - on `X_test` (`transform`), so that no information from the test set (category frequencies, scaling statistics) leaks into the preprocessing.

In [7]:
categorical_cols = ["gender", "smoking_history"]
robust_cols = ["bmi", "blood_glucose_level"]
standard_cols = ["age", "HbA1c_level"]

In [8]:
preprocessor = make_column_transformer(
    (OneHotEncoder(), categorical_cols),
    (RobustScaler(), robust_cols),
    (StandardScaler(), standard_cols),
    remainder="passthrough",
    verbose_feature_names_out=False
)

In [9]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [10]:
feature_names = preprocessor.get_feature_names_out()
X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

In [11]:
X_train_transformed

,gender_Female,gender_Male,smoking_history_No Info,smoking_history_current,smoking_history_never,smoking_history_past_smoker,bmi,blood_glucose_level,age,HbA1c_level,hypertension,heart_disease
12239,1.0,0.0,0.0,0.0,1.0,0.0,-0.494590,-0.677966,-0.879517,0.528048,0.0,0.0
50857,1.0,0.0,0.0,0.0,0.0,1.0,0.174652,2.033898,1.210443,3.231685,0.0,1.0
61036,0.0,1.0,0.0,0.0,1.0,0.0,-1.027821,1.016949,-0.479312,0.994193,0.0,0.0
33310,0.0,1.0,0.0,0.0,1.0,0.0,-0.024730,0.338983,0.765771,2.485854,0.0,0.0
43742,1.0,0.0,0.0,0.0,1.0,0.0,-0.721793,0.000000,-0.790582,0.621277,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
48403,1.0,0.0,0.0,0.0,0.0,1.0,0.612056,-0.847458,-1.057385,0.528048,0.0,0.0
47363,1.0,0.0,1.0,0.0,0.0,0.0,1.911901,1.016949,-0.123574,-1.895902,0.0,0.0
52118,0.0,1.0,1.0,0.0,0.0,0.0,0.017002,0.338983,-1.057385,0.248362,0.0,0.0
27390,1.0,0.0,0.0,0.0,0.0,1.0,0.358578,2.372881,1.032574,0.248362,1.0,0.0


In [12]:
X_test_transformed

,gender_Female,gender_Male,smoking_history_No Info,smoking_history_current,smoking_history_never,smoking_history_past_smoker,bmi,blood_glucose_level,age,HbA1c_level,hypertension,heart_disease
699,1.0,0.0,0.0,0.0,1.0,0.0,0.593509,0.322034,-0.079107,0.994193,0.0,0.0
64298,1.0,0.0,1.0,0.0,0.0,0.0,-0.879444,0.000000,0.765771,-1.429758,1.0,0.0
56235,1.0,0.0,1.0,0.0,0.0,0.0,0.302937,-0.169492,0.721303,0.621277,0.0,0.0
32411,0.0,1.0,1.0,0.0,0.0,0.0,0.106646,0.254237,-0.345910,-0.963613,0.0,0.0
62655,1.0,0.0,0.0,0.0,1.0,0.0,2.812983,0.305085,-0.746115,0.248362,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1860,1.0,0.0,1.0,0.0,0.0,0.0,-0.845440,-1.016949,0.232164,-1.429758,0.0,0.0
90597,1.0,0.0,0.0,0.0,0.0,1.0,-0.268934,0.254237,1.432779,0.155133,0.0,0.0
91673,1.0,0.0,0.0,0.0,1.0,0.0,1.814529,-0.847458,-0.657180,0.900964,0.0,0.0
51447,1.0,0.0,1.0,0.0,0.0,0.0,-0.276662,0.305085,0.054295,0.900964,0.0,0.0


## Save Datasets

In [ ]:
X_train_transformed.to_csv("../data/processed/diabetes_data_train.csv", index=False)

In [ ]:
X_test_transformed.to_csv("../data/processed/diabetes_data_test.csv", index=False)

In [15]:
y_train.to_csv("../data/processed/diabetes_data_y_train.csv", index=False)

In [16]:
y_test.to_csv("../data/processed/diabetes_data_y_test.csv", index=False)